# Update Climate Indices

This assumes you have downloaded the climate files.  To do so,  in /home/joe/Fire/ML/DB/Climate-Indices/New, there is a file called list.  run: wget -i list to retrieve teh files

In [3]:
import pandas as pd
import glob
import os

# 1. Define the path where you saved your wget files
path = '/home/joe/work/Fire/ML/Data/Climate-Indices/'
all_files = glob.glob(os.path.join(path, "*.data"))  # Change extension if needed (.txt, .tsv, etc.)
DB_PATH_NEW = os.path.join(path, "New", "climate_indices.db")


In [ ]:

# 2. Use a loop to read them in and keep only rows that start with a 4-digit year
data_frames = []

for filename in all_files:
    df = pd.read_csv(
        filename,
        sep=r"\s+",
        header=None,
        skiprows=1,
        engine="python",
        on_bad_lines="skip",
        dtype=str,
    )

    year_text = df[0].astype(str).str.strip()
    year_mask = year_text.str.fullmatch(r"\d{4}")
    df = df.loc[year_mask].copy()
    df[0] = year_text.loc[year_mask].astype(int)

    # Convert the remaining columns back to numeric where possible.
    for column in df.columns[1:]:
        df[column] = pd.to_numeric(df[column], errors="coerce")

    df['source_file'] = os.path.basename(filename)
    data_frames.append(df)

# 3. Merge everything into one DataFrame
df_master = pd.concat(data_frames, ignore_index=True)

print(f"Successfully loaded {len(data_frames)} files.")

value_cols = list(range(1, 13))
dataFramesOut=[]
for df in data_frames:
    file = df['source_file'].unique()[0]
    df[0] = pd.to_numeric(df[0], errors="coerce")  
    df = df.loc[df[0].between(1990, 2026)]    
    count = (df[value_cols] < -99).sum().sum()   
    if count > 9:
        continue    
    if df[0].max() < 2026 or df[0].min() < 1990:
        continue
    print(file, count, df[0].min(), df[0].max(), df.isna().sum().sum())
    dataFramesOut.append(df)

In [ ]:
long_frames = []

for df in dataFramesOut:
    index_name = os.path.splitext(df['source_file'].unique()[0])[0].replace(".", "_")

    # Melt the 12 month columns (1–12) into rows
    melted = df.melt(id_vars=[0], value_vars=list(range(1, 13)),
                     var_name="month", value_name=index_name)

    # Build YYYYMM key
    melted["yrmo"] = melted[0].astype(int) * 100 + melted["month"].astype(int)

    # Replace missing-value flags with NaN
    melted.loc[melted[index_name] < -99, index_name] = float("nan")

    melted = melted[["yrmo", index_name]].sort_values("yrmo").reset_index(drop=True)
    long_frames.append(melted)

# Merge all indices on yrmo
from functools import reduce
idx_new = reduce(lambda a, b: a.merge(b, on="yrmo", how="outer"), long_frames)
idx_new = idx_new.sort_values("yrmo").reset_index(drop=True)

# Remove months where every index value is missing
value_cols = [c for c in idx_new.columns if c != "yrmo"]
idx_new = idx_new.dropna(subset=value_cols, how="all").reset_index(drop=True)

print(idx_new.shape)
print(idx_new.head())

# Persist to the new DB
with sqlite3.connect(DB_PATH_NEW) as conn:
    idx_new.to_sql("indices", conn, if_exists="replace", index=False)
    print(f"\nWrote indices table ({len(idx_new)} rows × {len(idx_new.columns)} cols) → {DB_PATH_NEW}")

## Check

In [4]:

import sqlite3



conn = sqlite3.connect(DB_PATH_NEW)
idx = pd.read_sql(f"SELECT * FROM indices", conn)
conn.close()

idx.isna().sum()


yrmo           0
wp             0
nina4          0
nao            0
nina34         0
ea             0
soi            0
ao             0
nina1_anom     0
nina34_anom    0
nina3          0
nina1          0
pna            0
nina4_anom     0
nina3_anom     0
dtype: int64

In [5]:
idx.tail()

,yrmo,wp,nina4,nao,nina34,ea,soi,ao,nina1_anom,nina34_anom,nina3,nina1,pna,nina4_anom,nina3_anom
430,202511,0.47,28.17,-0.96,26.01,0.29,1.8,-0.467,-0.46,-0.70,24.47,21.19,0.74,-0.52,-0.63
431,202512,0.08,28.21,-0.69,25.93,-0.18,0.0,0.267,-0.62,-0.67,24.43,22.19,-1.71,-0.33,-0.80
432,202601,0.07,28.24,-0.93,25.96,0.50,1.8,-2.048,-0.29,-0.58,25.02,24.28,0.36,-0.08,-0.64
433,202602,0.23,28.40,0.33,26.48,2.07,2.4,-1.256,0.72,-0.27,26.31,26.82,-1.04,0.20,-0.10
434,202603,-2.13,28.67,2.42,27.23,-0.77,2.0,2.044,0.82,-0.06,27.41,27.31,-1.99,0.35,0.20
